## Bi-Encoder 

- With hyperparameter search 
- For both GloVe and BERT word embeddings
- Using a `keras.Model` subclass

In [1]:
# -----MAIN HYPERPARAMS-------
EMBEDDINGS = 'bert'
BATCH_SIZE = 128
NAME = 'tradeoff'
DATASET_SIZE = 'small'
# ----------------------------

from recs import *
from itertools import product

data = DataLoader(BATCH_SIZE, data_size=DATASET_SIZE)
if EMBEDDINGS == 'glove':
    TRAIN_EMBEDS = False
    MAX_TITLE_LENGTH = 20    
    vectorizer = layers.TextVectorization(
        max_tokens=20000, output_sequence_length=MAX_TITLE_LENGTH)
    train_ds = data.get_train_ds(vectorizer) 

else:
    TRAIN_EMBEDS = False
    MAX_TITLE_LENGTH = 1  
    train_ds = data.get_train_ds()
    
val_ds = data.get_val()
data.get_popularity() # load popularity for novelty regularisation

Available device: GPU 



### Define Hyperparams

Overview of loss ranges and suggested weight (lambda) ranges:

Param | weight | loss range
---|---|---
Main (rank) loss | Main |+/-850
lambda_diversity | 100 | 30
lambda_novelty | 10 | -70

In [2]:
hps = dict(    
    dropout=[0.2], 
    head_num=[4], # nmrs = 16
    head_dim=[64], 
    intermediate_dim=[128],
    lr=[5e-4],  
    # lambda_diversity=list(np.arange(50, 201, 50)), 
    lambda_novelty=[0], #list(np.arange(8, 14.1, 2)),   # Only for Bert
    )

print(f"hps contain {len(list(product(*hps.values())))} unique permutations") 
pd.DataFrame(list(product(*hps.values())), columns=hps.keys())

hps contain 1 unique permutations


,dropout,head_num,head_dim,intermediate_dim,lr,lambda_novelty
0,0.2,4,64,128,0.0005,0


In [3]:
def get_model(hyperparams):
    """Gets the model and loads global hyperparams"""
    lr = hyperparams.pop('lr')
    model = OptimisedModel(
        data=data,        
        embeddings=EMBEDDINGS,
        train_embeds=TRAIN_EMBEDS,
        dataset_size=DATASET_SIZE,
        batch_size=BATCH_SIZE,
        max_title_length=MAX_TITLE_LENGTH,
        **hyperparams)

    model.compile(optimizer=keras.optimizers.Adam(lr),
                loss=RankLoss(), metrics=[MaskedAUC()])
    
    return model

def train(model):
    """Train model and return history"""
    return model.fit(
        train_ds, 
        validation_data=val_ds,
        epochs=10, 
        verbose=1,
        callbacks=[
            keras.callbacks.EarlyStopping(monitor='val_masked_auc', mode='max')],
        steps_per_epoch=data.steps_per_epoch,
        validation_steps=data.validation_steps)

In [4]:
def test():
    loss_fn = RankLoss()
    model = get_model({
        'dropout': 0.2, 
        'head_num': 4, 
        'head_dim': 64,
        'intermediate_dim': 64,
        'lr': 0.001,
        'lambda_diversity': 100,
        'lambda_novelty': 10})

    inputs, labels = list(train_ds.take(1))[0]    
    predictions = model(inputs, training=True)        
    rank_loss = loss_fn(labels, predictions)
    regularisation = model.compute_loss(y=labels, y_pred=predictions)
    
    print(f"Weighted regularisation: {regularisation.numpy()} \
            \nRank loss: {rank_loss}")        
test()

Diversity: 47.01155471801758
Surprisal: -72.92535400390625
Weighted regularisation: 1128.5760498046875             
Rank loss: 1154.48974609375


In [5]:
stop

NameError: name 'stop' is not defined


---

### Random search

In [ ]:
# random search
n_trials = 4

def sample_hparams(hps, n_trials=10):
    """sample unique parameters to try"""
    samples = []   

    # Don't exceed the number of possible permutations
    n_trials = min(n_trials, len(list(product(*hps.values()))))
    n = 0
    while n < n_trials:
        sample = {k: np.random.choice(v) for k, v in hps.items()}

        # Only add unknown combinations
        if sample not in samples:
            n += 1
            samples.append(sample)   
    return samples    

sampled_hps = sample_hparams(hps, n_trials)
pd.DataFrame(sampled_hps)

,dropout,head_num,head_dim,intermediate_dim,lr,lambda_diversity
0,0.2,4,64,128,0.0005,450
1,0.2,4,64,128,0.0005,300
2,0.2,4,64,128,0.0005,400
3,0.2,4,64,128,0.0005,50


In [ ]:
logs = []

for i, hyperparams in enumerate(sampled_hps): 
    log = {'hyperparams': {k: float(v) for k,v in hyperparams.items()}}        
    print(f"\n{'-'*40}Trail: {i+1}/{len(sampled_hps)}{'-'*40} \n")
    display(pd.DataFrame([hyperparams]))
    print(f"\n{'-'*100}\n")
    model = get_model(hyperparams)
    history = train(model)
    
    metrics = {k: history.history[k][-1] for k in history.history.keys()}
    log.update({'results': metrics})
    logs.append(log)
    
    y_pred = model.predict(data.get_test())
    log_results(y_true=data.test_labels, 
                y_pred=y_pred, 
                imprs=data.test_imprs, 
                modelname=f'{NAME}_{EMBEDDINGS}_{i+1}', 
                history=history,
                hyperparams=log['hyperparams'], 
                dataset_size=DATASET_SIZE,
                )

    del model

In [ ]:
# best hyperparams
pd.DataFrame([log['hyperparams'] for log in logs], 
             index=[log['results']['val_masked_auc'] for log in logs]
             ).round(2).sort_index(ascending=False)

,dropout,head_num,head_dim,intermediate_dim,lr,lambda_diversity,lambda_novelty
0.688922,0.2,4.0,64.0,64.0,0.0,10000.0,510.0
0.686403,0.2,4.0,64.0,64.0,0.0,5000.0,510.0
0.670937,0.2,4.0,64.0,64.0,0.0,5000.0,710.0
0.663587,0.2,4.0,64.0,64.0,0.0,5000.0,210.0
0.499741,0.2,4.0,64.0,64.0,0.0,10000.0,710.0



---

### Grid search

In [6]:
# Grid search
repeats = 2

grid = list(product(*hps.values()))
logs = []
for i, param_set in enumerate(grid):
    for j in range(repeats):
        hyperparams = {k: p for k,p in zip(hps.keys(), param_set)}
        log = {'hyperparams': {k: float(v) for k,v in hyperparams.items()}}    
        print(f"\n{'-'*30}Trail: {i+1}/{len(grid)} (repeats: {j+1}/{repeats}){'-'*30} \n")
        display(pd.DataFrame([hyperparams]))
        print(f"\n{'-'*100}\n")

        model = get_model(hyperparams)
        history = train(model)
        metrics = {k: history.history[k][-1] for k in history.history.keys()}
        log.update({'results': metrics, 'epochs': history.epoch})
        logs.append(log)
        
        y_pred = model.predict(data.get_test())
        log_results(y_true=data.test_labels, 
                    y_pred=y_pred, 
                    imprs=data.test_imprs, 
                    modelname=f'{NAME}_{EMBEDDINGS}_v1.{i+1}.{j}', 
                    history=history,
                    hyperparams=log['hyperparams'], 
                    dataset_size=DATASET_SIZE
                    )
        del model


------------------------------Trail: 1/1 (repeats: 1/2)------------------------------ 



,dropout,head_num,head_dim,intermediate_dim,lr,lambda_novelty
0,0.2,4,64,128,0.0005,0



----------------------------------------------------------------------------------------------------

Epoch 1/10


I0000 00:00:1743612703.856240  335548 service.cc:146] XLA service 0x7159540036e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1743612703.856262  335548 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce RTX 4070 SUPER, Compute Capability 8.9


   3/1226 ━━━━━━━━━━━━━━━━━━━━ 45s 37ms/step - loss: 1047.8149 - masked_auc: 0.5070     

...

,tradeoff_bert_v1.1.0
modelname,tradeoff_bert_v1.1.0
dataset_size,small
timestamp,2025-04-02 17:56
auc,0.7194
mean_mrr,0.2916
ndcg@5,0.3191
ndcg@10,0.3809
mean_epc,1.392
mean_intra_list_diversity,0.2423
mean_surprisal,7.884



------------------------------Trail: 1/1 (repeats: 2/2)------------------------------ 



,dropout,head_num,head_dim,intermediate_dim,lr,lambda_novelty
0,0.2,4,64,128,0.0005,0



----------------------------------------------------------------------------------------------------

Epoch 1/10
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 208s 72ms/step - loss: 957.3903 - masked_auc: 0.7001 - val_loss: 985.3207 - val_masked_auc: 0.7098
Epoch 2/10
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 52s 42ms/step - loss: 941.9163 - masked_auc: 0.7397 - val_loss: 983.3887 - val_masked_auc: 0.7157
Epoch 3/10
1226/1226 ━━━━━━━━━━━━━━━━━━━━ 52s 42ms/step - loss: 939.5979 - masked_auc: 0.7470 - val_loss: 982.6857 - val_masked_auc: 0.7156
572/572 ━━━━━━━━━━━━━━━━━━━━ 82s 77ms/step
Saved model predictions here: ../.data/tradeoff_bert_v1.1.1.npy


,tradeoff_bert_v1.1.1
modelname,tradeoff_bert_v1.1.1
dataset_size,small
timestamp,2025-04-02 18:03
auc,0.7229
mean_mrr,0.2817
ndcg@5,0.309
ndcg@10,0.3739
mean_epc,1.3819
mean_intra_list_diversity,0.2549
mean_surprisal,7.7181



---


In [ ]:
loss_fn = RankLoss()
inputs, labels = list(train_ds.take(1))[0]
predictions = model(inputs, training=True)        
rank_loss = loss_fn(labels, predictions)
regularisation = model.compute_loss(y=labels, y_pred=predictions)

print(f"Weighted regularisation: {regularisation.numpy()} \
        \nRank loss: {rank_loss}")

Diversity: 76.05571746826172
Weighted regularisation: 2166.26904296875         
Rank loss: 2090.21337890625
